# Activity 2: Pandas to SQL Bridge

Every filter and join you write in pandas has a **SQL twin**. Today you learn the pandas
version; on Day 5 you write the SQL. Each section shows the SQL equivalent right next to the
pandas code, so the transition is a translation, not a new language.

Work one cell at a time: read the worked example, run it, then do the challenge below it.

Solutions notebook for instructor reference. Keep the student drill notebook unchanged.

## Instructor Teaching Note

Teach the boolean mask first because students can see each condition directly. Then show `.query()` as a readable alternate that feels closer to SQL. The main lesson is that pandas and SQL often express the same idea with different syntax.

## Setup: two small tables

`orders` is a fact table; `customers` is a dimension. Same shape you will meet in SQL.

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "order_id":    [1, 2, 3, 4, 5, 6, 7, 8],
    "customer_id": [10, 10, 11, 12, 11, 99, 12, 10],
    "amount":      [120, 45, 210, 60, 30, 150, 500, 90],
    "order_date":  pd.to_datetime([
        "2026-01-05","2026-01-18","2026-02-02","2026-02-14",
        "2026-03-01","2026-03-09","2026-03-20","2026-04-02"]),
    "status": ["paid","paid","refunded","paid","cancelled","paid","shipped","refunded"],
})
customers = pd.DataFrame({
    "customer_id": [10, 11, 12, 13],
    "name": ["Ana","Ben","Cy","Dee"],
    "city": ["Hartford","Boston","Hartford","Miami"],
})
orders

## 1. Filtering rows = SQL `WHERE`

Two pandas ways: a **boolean mask** `df[df["col"] > x]`, or **`.query()`** which reads almost like SQL.

```sql
SELECT * FROM orders WHERE amount > 100;
```

In [ ]:
# boolean mask
orders[orders["amount"] > 100]

In [ ]:
# same result with .query() -- closest to SQL
orders.query("amount > 100")

**Challenge 1.** Return orders where `amount` is 50 or less. Try it both ways.

`SQL: WHERE amount <= 50`

In [ ]:
# Simple path: boolean mask.
orders[orders["amount"] <= 50]

# Also valid: .query() reads more like SQL.
# orders.query("amount <= 50")

## 2. Combining conditions = `AND` / `OR` / `NOT`

In a pandas mask use `&` `|` `~` (not the words `and`/`or`/`not`), and **wrap each condition in
parentheses**. In `.query()` you can use the words.  

| pandas mask | .query() | SQL |
|---|---|---|
| `&` | `and` | `AND` |
| `|` | `or` | `OR` |
| `~` | `not` | `NOT` |

In [ ]:
# AND: over 100 AND paid
orders[(orders["amount"] > 100) & (orders["status"] == "paid")]

In [ ]:
# same with query
orders.query("amount > 100 and status == 'paid'")

**Challenge 2.** Return orders where `amount > 200` **OR** `status` is `refunded`.

`SQL: WHERE amount > 200 OR status = 'refunded'`

In [ ]:
# Simple path: boolean mask with | for OR.
orders[(orders["amount"] > 200) | (orders["status"] == "refunded")]

# Also valid: .query() can use the word "or".
# orders.query("amount > 200 or status == 'refunded'")

## 3. Ranges = SQL `BETWEEN`

`Series.between(low, high)` is inclusive on both ends, exactly like SQL `BETWEEN`. Works on
numbers **and** dates.

```sql
SELECT * FROM orders WHERE amount BETWEEN 50 AND 150;
```

In [ ]:
# numeric range
orders[orders["amount"].between(50, 150)]

In [ ]:
# date range -- same method
orders[orders["order_date"].between("2026-02-01", "2026-03-15")]

**Challenge 3.** Return orders placed in the **first quarter** (Jan 1 to Mar 31, 2026).

`SQL: WHERE order_date BETWEEN '2026-01-01' AND '2026-03-31'`

In [ ]:
orders[orders["order_date"].between("2026-01-01", "2026-03-31")]

## 4. Membership = `IN` and `NOT IN`

`Series.isin([...])` is SQL `IN`. Put `~` in front for `NOT IN`.

```sql
SELECT * FROM orders WHERE status IN ('paid','shipped');
```

In [ ]:
# IN
orders[orders["status"].isin(["paid", "shipped"])]

In [ ]:
# NOT IN: the ~ flips the mask
orders[~orders["status"].isin(["cancelled", "refunded"])]

**Challenge 4.** From `customers`, return everyone **not** in Hartford.

`SQL: WHERE city NOT IN ('Hartford')` (or `city <> 'Hartford'`)

In [ ]:
# Simple path: use ~ to flip the isin() match.
customers[~customers["city"].isin(["Hartford"])]

# Also valid for one value:
# customers[customers["city"] != "Hartford"]

## 5. Exclude a value = `!=` / `<>`

For a single value, `!=` is clearest. `~` is for flipping a whole mask (like `~isin(...)` above).

```sql
SELECT * FROM orders WHERE status <> 'cancelled';
```

In [ ]:
orders[orders["status"] != "cancelled"]
# or: orders.query("status != 'cancelled'")

## 6. Combining tables = SQL `JOIN`

`pd.merge(left, right, on=key, how=...)` is your workhorse join. `how` maps straight to SQL:
`inner`, `left`, `right`, `outer`.

```sql
SELECT * FROM orders o JOIN customers c ON o.customer_id = c.customer_id;
```

In [ ]:
# INNER JOIN: only rows with a match in both
pd.merge(orders, customers, on="customer_id", how="inner")

**Watch the row count.** On Day 5 you will check row counts before and after every join,
because a join can add or drop rows. `orders` has 8 rows; customer_id 99 has no match.

In [ ]:
# LEFT JOIN keeps ALL orders, even the unmatched one (city becomes NaN)
left = pd.merge(orders, customers, on="customer_id", how="left")
print("orders:", len(orders), " after left join:", len(left))
left

**Challenge 5 (anti-join).** Find the orders that have **no matching customer**.
Hint: left join with `indicator=True`, then keep rows where `_merge == 'left_only'`.

`SQL: LEFT JOIN ... WHERE c.customer_id IS NULL`

In [ ]:
merged = pd.merge(orders, customers, on="customer_id", how="left", indicator=True)
merged[merged["_merge"] == "left_only"]

## 7. Stacking rows = `UNION ALL`

`pd.concat([...])` stacks rows from tables with the same columns, like SQL `UNION ALL`
(keeps duplicates; use `.drop_duplicates()` for a plain `UNION`).

In [ ]:
more_orders = pd.DataFrame({
    "order_id": [9, 10], "customer_id": [11, 13], "amount": [75, 300],
    "order_date": pd.to_datetime(["2026-04-10","2026-04-15"]),
    "status": ["paid","paid"],
})
all_orders = pd.concat([orders, more_orders], ignore_index=True)
print("rows:", len(all_orders))

## 8. `DataFrame.join` (join on the index)

`merge` joins on columns; `DataFrame.join` joins on the **index**. Handy when the key is already
the index. Same JOIN idea, different entry point.

In [ ]:
orders_i = orders.set_index("customer_id")
cust_i = customers.set_index("customer_id")
orders_i.join(cust_i, how="left").head()

## Cheat sheet: pandas ↔ SQL

| Goal | pandas | SQL |
|---|---|---|
| filter | `df[df.x > 5]` / `df.query("x > 5")` | `WHERE x > 5` |
| and / or / not | `&` `|` `~` (mask) | `AND` `OR` `NOT` |
| range | `df.x.between(a, b)` | `BETWEEN a AND b` |
| membership | `df.x.isin([...])` / `~...isin` | `IN` / `NOT IN` |
| exclude value | `df[df.x != v]` | `x <> v` |
| join | `pd.merge(a, b, on=k, how=...)` | `JOIN ... ON` |
| stack rows | `pd.concat([a, b])` | `UNION ALL` |

On Day 5 you will write the right-hand column by hand. Today you built the intuition on the left.